# Laboratorio 4

## Integrantes

| Nombre                            | Carnet | Usuario Git |
| --------------------------------- | ------ | ----------- |
| Edwin Jose Gabriel De Leon Garcia | 22809  | EJGDLG      |
| Gustavo Adolfo Cruz Bardales      | 22779  | G2309       |
| Josué Emanuel Say Garcia          | 22801  | JosueSay    |
| Mathew Alexander Cordero Aquino   | 22982  | donmatthiuz |


## Repositorio

[Link al Repositorio](https://github.com/donmatthiuz/RL/tree/lab4)

## Caso

Una empresa de logística de última milla está evaluando el uso de robots autónomos para la gestión interna de su almacén principal. El almacén se modela como una cuadrícula de 8x8 con puntos de recogida, puntos de entrega, zonas de penalización por congestión, y obstáculos fijos. El equipo de ingeniería necesita comparar dos estrategias de aprendizaje antes de comprometer recursos en un sistema completo: una política conservadora que aprenda a navegar de forma segura durante el entrenamiento, y una política agresiva que busque la ruta óptima sin importar los riesgos durante la exploración.

Su grupo ha sido contratado para implementar ambas estrategias usando SARSA y Q-Learning respectivamente, comparar su comportamiento empírico, y producir un dictamen técnico con recomendaciones concretas para la gerencia.

## Task 1

Diseñen formalmente el MDP que representa el almacén. El diseño debe especificar:

### Inciso 1

El espacio de estados y el espacio de acciones. Justifiquen cada decisión considerando la interfaz de Gymnasium: `observation_space` y `action_space` deben ser instancias de `gymnasium.spaces.Discrete` o `gymnasium.spaces.Box` según corresponda. Argumenten cuál es más apropiado para este dominio.

**Respuestas:**

**Espacio de estados $\mathcal{S}$:**

El almacén es una cuadrícula de $8\times8$. El estado debe representar la posición actual del robot y si lleva o no un paquete:

$
s=(fila,columna,carga)
$

donde $fila,columna\in{0,\ldots,7}$ y $carga\in{0,1}$.

Se utilizará `gymnasium.spaces.Discrete(128)`, ya que existen:

$
8\times8\times2=128
$

combinaciones posibles porque el entorno tiene un número finito y discreto de estados, en caso de ser variables continuas se usaría box.

**Espacio de acciones $\mathcal{A}$:**

$
\mathcal{A}={\text{arriba},\text{abajo},\text{izquierda},\text{derecha}}
$

Se utilizará:

```python
action_space = gymnasium.spaces.Discrete(4)
```

Cada valor representa una dirección, por ejemplo:

$
0=\uparrow,\quad1=\rightarrow,\quad2=\downarrow,\quad3=\leftarrow
$

`Discrete(4)` porque el robot tiene 4 acciones.

### Inciso 2

La función de recompensa con al menos tres componentes: recompensa por entrega exitosa, penalización por zona de congestión, y penalización por paso. Justifiquen la magnitud relativa de cada componente y argumenten qué comportamiento indeseable produciría una ponderación incorrecta de alguno de ellos.

**Respuestas:**

Se propone la siguiente función de recompensa:

$$
r(s,a,s')=
\begin{cases}
+100 & \text{si realiza una entrega exitosa}\
-10 & \text{si entra en una zona de congestión}\
-1 & \text{por cada paso realizado}
\end{cases}
$$

Una entrega recibe la mayor recompensa porque representa el objetivo principal ($+100$). La congestión tiene una penalización mayor que un paso normal $(-10)$ para incentivar rutas seguras, mientras que cada paso cuesta $-1$ para favorecer rutas cortas.

Una ponderación incorrecta se observaría en penalización de congestión demasiado pequeña haría que el robot la ignore, mientras que una penalización excesiva podría provocar rutas innecesariamente largas. Si el costo por paso fuera demasiado alto, el agente podría preferir atravesar zonas riesgosas únicamente para reducir la distancia.

### Inciso 3

La condición de terminación del episodio. ¿Cuándo termina un episodio? ¿Es apropiado tener un límite máximo de pasos? Justifiquen.

**Respuestas:**

El episodio termina cuando el robot completa la entrega en el punto.

Sí es adecuado colocar un límite de pasos como $T_{\max}=200$

porque si el robot alcanza los $200$ pasos sin completar la entrega puede indicar que esta truncado. Este límite es apropiado porque evita episodios largos causados por exploración, ciclos. Además, permite mantener un costo computacional controlado durante el entrenamiento. En Gymnasium, la entrega corresponde a `terminated=True`, mientras que alcanzar el límite de pasos corresponde a `truncated=True`.

### Inciso 4

El diseño del mapa $8 \times 8$: ubiquen al menos dos zonas de congestión adyacentes a rutas de alta recompensa. Esta configuración específica es crítica para que la comparación entre SARSA y Q-Learning sea informativa. Expliquen por qué esa configuración espacial genera el comportamiento diferencial esperado entre ambos algoritmos.

**Respuestas:**

```text
- - - - - - - -
- R - - - - - -
- - # # - - - -
- - - - - - - -
- - - C C C E -
- - - - - - - -
- - # # - - - -
- - - - - - - -
```

Donde:

* `R` = punto de recogida.
* `E` = punto de entrega.
* `C` = zona de congestión.
* `#` = obstáculo.
* `-` = espacio libre.

La ubicación de congestión cerca de una ruta atractiva genera dos alternativas:

- Ruta corta pero riesgosa.
- Ruta más larga pero segura.

**SARSA**, al ser *on-policy*, aprende considerando las acciones exploratorias que realmente ejecuta, por lo que tiende a valorar más las rutas seguras. **Q-Learning**, al ser *off-policy*, actualiza utilizando la mejor acción futura:

$$
\max_{a'}Q(s',a')
$$

por lo que tiende a aprender la ruta óptima más corta aunque esté próxima a zonas penalizadas.

## Task 2

Antes de implementar nada, respondan las siguientes preguntas con argumentación técnica:

### Inciso 1

Para el entorno que diseñaron, predigan formalmente cuál algoritmo, SARSA o Q-Learning, produciría mayor recompensa acumulada durante el entrenamiento y cuál producirá mayor recompensa durante la evaluación con política greedy pura. Justifiquen cada predicción usando las propiedades on-policy y off-policy de cada algoritmo y la estructura específica de su mapa.

**Respuesta:**

SARSA debería obtener una mayor recompensa acumulada porque es *on-policy* y actualiza considerando la acción $a'$ que realmente ejecutará:

$$
Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma Q(s',a')-Q(s,a)]
$$

Como la política $\epsilon$-greedy puede cometer acciones exploratorias y SARSA aprende el riesgo asociado a transitar cerca de las zonas de congestión y tiende a elegir la ruta más segura reduciendo penalizaciones en el entrenamiento

Q-Learning al ser *off-policy*, actualiza suponiendo que en el siguiente estado tomará la mejor acción:

$$
Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]
$$

Por ello puede aprender una ruta más corta cercana a la congestión, aunque durante la exploración tenga más penalizaciones.

Durante la **evaluación con política greedy pura** $(\epsilon=0)$, se espera que Q-Learning obtenga mayor recompensa porque ya no existe el riesgo de acciones exploratorias y puede explotar directamente la ruta óptima aprendida. SARSA puede conservar una ruta más segura pero más larga, acumulando más penalizaciones de $-1$ por paso.

### Inciso 2

Argumenten cómo afecta el valor de $\epsilon$ al comportamiento diferencial entre SARSA y Q-Learning en su entorno. ¿Existe un valor de $\epsilon$ para el cual ambos algoritmos convergen a políticas idénticas? Justifiquen matemáticamente.

**Respuesta:**

En una política $\epsilon$-greedy, con $|\mathcal A|=4$:

$$
P(a^*|s)=1-\epsilon+\frac{\epsilon}{4}
$$

y para cada acción no greedy:

$$
P(a|s)=\frac{\epsilon}{4}
$$

Cuando $\epsilon$ es alto, hay mayor exploración. Esto aumenta la diferencia entre los algoritmos. SARSA incorpora indirectamente ese riesgo exploratorio en sus actualizaciones, mientras que Q-Learning continúa actualizando respecto a $\max_{a'}Q(s',a')$.

Cuando:

$$
\epsilon\rightarrow0
$$

la política de comportamiento de SARSA se aproxima a una política greedy:

$$
Q(s',a')\rightarrow\max_{a'}Q(s',a')
$$

Con $\epsilon=0$ y con condiciones de convergencia adecuados tanto SARSA y Q-Learning pueden converger a la misma política óptima. Porque durante el aprendizaje normalmente se utiliza un $\epsilon>0$ porque se quiere garantizar suficiente exploración.

### Inciso 3

Para su función de recompensa específica, calculen una cota superior del valor óptimo $V^*(s_0)$ del estado inicial, asumiendo que el agente siempre toma la ruta más corta sin pasar por zonas de congestión. Usen esta cota como referencia para evaluar qué tan cerca llegan sus implementaciones al óptimo teórico.

**Respuesta:**

Función de recompensa:

* Entrega exitosa: $+100$.
* Paso normal: $-1$.
* Congestión: $-10$.
* Factor de descuento: $\gamma=0.95$.

Si la ruta segura más corta desde $s_0$ hasta la entrega requiere $d$ movimientos y no atraviesa ninguna congestión, la cota de referencia es:

$$
V^*(s_0)\leq
-\sum_{t=0}^{d-2}\gamma^t
+100\gamma^{d-1}
$$

Usando la suma geométrica:

$$
V^*(s_0)\leq
-\frac{1-\gamma^{d-1}}{1-\gamma}
+100\gamma^{d-1}
$$

Sustituyendo $\gamma=0.95$:

$$
\boxed{
V^*(s_0)\leq
-\frac{1-0.95^{d-1}}{0.05}
+100(0.95)^{d-1}
}
$$

Cuanto más cercano sea el retorno obtenido por SARSA o Q-Learning a este valor, más próxima estará su política al óptimo teórico.


## Task 3


Implementen el entorno y los algoritmos con las siguientes especificaciones:

El entorno debe implementarse como una clase `WarehouseEnv` que herede de `gymnasium.Env`, con los métodos `reset`, `step` y `render` correctamente implementados según la interfaz estándar de Gymnasium. El método `render` debe producir una visualización en consola o matplotlib del estado actual del agente en el mapa.

Los algoritmos SARSA y Q-Learning deben implementarse desde cero como funciones o clases separadas que reciban el entorno como argumento y devuelvan la tabla Q aprendida y el historial de recompensas por episodio. No se permite usar implementaciones preconstruidas de SARSA o Q-Learning de ninguna librería.

La implementación debe registrar para cada episodio: la recompensa total acumulada, el número de pasos hasta terminación, y el número de veces que el agente pasó por zonas de congestión. Estos registros son necesarios para el análisis de la Tarea 4.

Al finalizar el entrenamiento, evalúen cada política aprendida con 100 episodios usando política greedy pura, sin exploración, y registren las mismas métricas.

## Task 4

Con base en lo aprendido, responda:

### Inciso 1

**Verificación de predicciones:** contrasten cada predicción de la Tarea 2 con los resultados observados. Para cada predicción indiquen si fue confirmada, refutada o inconclusa, y expliquen la discrepancia si la hay. Una predicción refutada con buena explicación vale más que una confirmada sin análisis.

### Inciso 2

**Análisis de convergencia y comportamiento diferencial:** grafiquen la recompensa promedio por episodio durante el entrenamiento para ambos algoritmos en la misma figura. Grafiquen también el número promedio de visitas a zonas de congestión por episodio. ¿En qué punto del entrenamiento se hace visible la diferencia entre SARSA y Q-Learning? ¿Coincide con lo esperado teóricamente?


### Inciso 3

**Análisis de sensibilidad:** repitan el experimento con al menos tres combinaciones de $(\alpha, \epsilon)$. Para cada combinación reporten la recompensa promedio en evaluación greedy. Construyan una tabla comparativa y argumenten cuál combinación recomendarían para producción y por qué.


### Inciso 4

**Investigación bibliográfica:** busquen y lean un paper publicado entre 2021 y 2025 que aplique Q-Learning, SARSA, o alguna extensión directa de TD Learning a un problema de navegación, robótica, o logística. El paper debe ser de una fuente indexada: NeurIPS, ICML, ICLR, JMLR, IEEE Robotics, o similar. Escriban un resumen técnico de media página que incluya: el problema que resuelve, cómo extiende o aplica TD Learning, los resultados principales, y una reflexión sobre qué limitaciones del TD tabular que discutimos esta semana resuelve ese trabajo y cuáles no.

### Inciso 5

**Dictamen técnico:** redacten un párrafo dirigido a la gerencia del almacén argumentando cuál de los dos algoritmos recomiendan para el sistema real, bajo qué condiciones operativas, y qué pasos adicionales serían necesarios antes de un despliegue en producción. El dictamen debe estar respaldado por evidencia de sus experimentos, no solo por argumentos teóricos.